# Punarvaani Sarvam-1 LoRA Fine-Tuning — Windows GPU (No Git Clone)

This notebook **does not clone or download the GitHub repository**. It lets you select the existing files from your Windows computer, then runs the current Sarvam fine-tuning function.

## Files you need locally
- `Sarvam.py` — the **current `Models/Sarvam.py` from the repo**. Do not use an old copy; the prompt tags and training fixes must match the repo.
- `train.jsonl` — training examples (cell 5 prints the exact count)
- `val.jsonl` — validation examples (cell 5 prints the exact count)

**Use this notebook with local Jupyter on Windows.** Run it from a local drive with about 10 GB free, for example `D:\punarvaani_sarvam`, not from OneDrive. It uses Windows file-selection dialogs. GPU training requires an NVIDIA GPU and CUDA-enabled PyTorch. The notebook cannot recreate the training function; it needs your existing `Sarvam.py`.

The exact row counts depend on the data you received; use the numbers printed by cell 5. The adapter this notebook produces is the handoff artifact. Copy only `Punarvaani_training/checkpoints/sarvam_lora/` back into the repo's `checkpoints/sarvam_lora/`.


## 1. Windows setup (run in PowerShell before opening this notebook)

Install Python 3.11, NVIDIA drivers, and Jupyter if not already installed.

```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
```

Install CUDA-enabled PyTorch from the official selector: https://pytorch.org/get-started/locally/

Then install notebook dependencies:
```powershell
pip install jupyter ipykernel
python -m ipykernel install --user --name punarvaani-gpu --display-name "Punarvaani GPU"
jupyter notebook
```

Select the **Punarvaani GPU** kernel. You may run this notebook from any folder; it will create a local `Punarvaani_training` folder for your files and outputs.


In [ ]:
# 2. Imports and GPU verification
import os, sys, json, shutil, subprocess, platform, importlib.util
from pathlib import Path

print("OS:", platform.platform())
print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA build:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
        subprocess.run(["nvidia-smi"], check=False)
    else:
        print("CUDA is not available. Install CUDA-enabled PyTorch and restart the kernel.")
except ImportError:
    raise RuntimeError("PyTorch missing. Install CUDA-enabled PyTorch using the official selector.")


## 3. Select the three project files

Run the cell below and choose:
1. Your existing `Sarvam.py`
2. `train.jsonl`
3. `val.jsonl`

The files will be copied into the notebook's working folder. If the file picker doesn't open (e.g. remote Jupyter), enter the paths manually in the fallback cell.


In [ ]:
# 3A. File picker (local Windows Jupyter)
from tkinter import Tk, filedialog

def choose_file(title):
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    path = filedialog.askopenfilename(title=title)
    root.destroy()
    if not path:
        raise FileNotFoundError(f"No file selected: {title}")
    return Path(path)

sarvam_source = choose_file("Select your existing Models/Sarvam.py")
train_source = choose_file("Select train.jsonl")
val_source = choose_file("Select val.jsonl")

print("Sarvam implementation:", sarvam_source)
print("Train dataset:", train_source)
print("Validation dataset:", val_source)


In [ ]:
# 3B. Fallback if file picker is unavailable:
# Uncomment and edit these paths, then run this cell.
# sarvam_source = Path(r"C:\Users\YOUR_NAME\Downloads\Sarvam.py")
# train_source = Path(r"C:\Users\YOUR_NAME\Downloads\train.jsonl")
# val_source = Path(r"C:\Users\YOUR_NAME\Downloads\val.jsonl")


In [ ]:
# 4. Prepare a local workspace (no repository clone)
WORK_DIR = Path.cwd() / "Punarvaani_training"
MODEL_DIR = WORK_DIR / "Models"
DATA_DIR = WORK_DIR / "sarvam_data"
OUTPUT_DIR = WORK_DIR / "checkpoints" / "sarvam_lora"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not sarvam_source.exists():
    raise FileNotFoundError(sarvam_source)
if not train_source.exists() or not val_source.exists():
    raise FileNotFoundError("Training or validation JSONL path does not exist.")

# Copy user-selected files into the workspace
shutil.copy2(sarvam_source, MODEL_DIR / "Sarvam.py")
shutil.copy2(train_source, DATA_DIR / "train.jsonl")
shutil.copy2(val_source, DATA_DIR / "val.jsonl")

print("Workspace:", WORK_DIR)
print("Sarvam.py:", MODEL_DIR / "Sarvam.py")
print("Train data:", DATA_DIR / "train.jsonl")
print("Validation data:", DATA_DIR / "val.jsonl")


In [ ]:
# 5. Validate JSONL counts and schema
# NOTE: this checks structure only. The phoneme strings must be generated
# with the repo's Data/hindi_g2p.py. A format-valid file with wrong G2P
# will train an adapter that does not match Stage 1.
import re

TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "val.jsonl"

EXPECTED_PREFIX = "<S2S>\n<PHONEMES>\n"
EXPECTED_SUFFIX = "\n</PHONEMES>\n<TEXT>\n"


def check_jsonl(path):
    count = 0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            obj = json.loads(line)
            if not {"prompt", "target"}.issubset(obj):
                raise ValueError(f"{path.name}, line {line_no}: expected keys 'prompt' and 'target'")

            prompt, target = obj["prompt"], obj["target"]
            if not isinstance(prompt, str) or not isinstance(target, str):
                raise ValueError(f"{path.name}, line {line_no}: prompt and target must be strings")
            if not prompt.startswith(EXPECTED_PREFIX):
                raise ValueError(f"{path.name}, line {line_no}: bad prompt start: {prompt[:80]!r}")
            if not prompt.endswith(EXPECTED_SUFFIX):
                raise ValueError(f"{path.name}, line {line_no}: bad prompt end: {prompt[-80:]!r}")
            if "\r" in prompt:
                raise ValueError(f"{path.name}, line {line_no}: prompt contains CRLF; regenerate with LF newlines")
            if not target.strip():
                raise ValueError(f"{path.name}, line {line_no}: empty target")
            if "<" in target or ">" in target:
                raise ValueError(f"{path.name}, line {line_no}: target contains tags: {target!r}")
            if re.search(r"[A-Za-z0-9]", target):
                raise ValueError(f"{path.name}, line {line_no}: target contains Latin letters/digits: {target!r}")
            count += 1
    return count


train_count = check_jsonl(TRAIN_PATH)
val_count = check_jsonl(VAL_PATH)
print("Train examples:", train_count)
print("Validation examples:", val_count)

with TRAIN_PATH.open(encoding="utf-8") as f:
    sample = json.loads(next(line for line in f if line.strip()))
print("\nPrompt sample (repr):", repr(sample["prompt"][:300]))
print("Target sample (repr):", repr(sample["target"][:200]))


## 6. Install the training dependencies

PyTorch should already be installed with CUDA support (section 1). Run the cell below once **before cell 7**; it installs the Sarvam training dependencies and pins `transformers` to the repo version. If your existing `Sarvam.py` imports other packages, install those too.


In [ ]:
# Run this once in the `punarvaani-gpu` kernel before cell 7.
# It does not reinstall PyTorch; install the CUDA PyTorch wheel first (see section 1).
%pip install "transformers==4.44.2" "peft==0.12.0" "accelerate>=0.33.0" "datasets>=2.20.0" sentencepiece protobuf tqdm


In [ ]:
# 7. Import your existing training function from the selected Sarvam.py
# It is loaded from the notebook-created workspace, not from GitHub.
import sys, importlib

sys.path.insert(0, str(WORK_DIR))
import Models.Sarvam as sarvam_module
importlib.reload(sarvam_module)

if not hasattr(sarvam_module, "train_sarvam_lora"):
    raise AttributeError(
        "The selected Sarvam.py does not define train_sarvam_lora(). "
        "Please choose the correct file containing your training function."
    )

train_sarvam_lora = sarvam_module.train_sarvam_lora
print("Loaded function from:", sarvam_module.__file__)


In [ ]:
# 7B. Verify the selected Sarvam.py matches the repo's prompt contract
EXPECTED_TAGS = ["<S2S>", "<PHONEMES>", "</PHONEMES>", "<TEXT>"]

actual_tags = getattr(sarvam_module, "TAGS", None)
if actual_tags != EXPECTED_TAGS:
    raise RuntimeError(
        f"Wrong Sarvam.py. Expected TAGS={EXPECTED_TAGS}, got {actual_tags}. "
        "Select the repo's current Models/Sarvam.py."
    )

expected_prompt = "<S2S>\n<PHONEMES>\nN A M A S TD E\n</PHONEMES>\n<TEXT>\n"
actual_prompt = sarvam_module.HindiPhonemeTranslator().build_prompt("N A M A S TD E")
if actual_prompt != expected_prompt:
    raise RuntimeError(
        "The selected Sarvam.py has a different prompt format. "
        f"Expected {expected_prompt!r}, got {actual_prompt!r}."
    )

print("Sarvam.py prompt contract matches the repo.")


In [ ]:
# 8. Training configuration
BASE_MODEL_ID = "sarvamai/sarvam-1"
EPOCHS = 2          # 60k noisy examples; use the noisy val loss to decide whether to continue.
BATCH_SIZE = 4      # A5000 24 GB should fit 4 at 512 tokens. If OOM, set 1.
GRAD_ACCUM = 1      # if BATCH_SIZE is 1, set this to 4 to keep the effective batch at 4.
LEARNING_RATE = 2e-4

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Training stopped.")

print("GPU:", torch.cuda.get_device_name(0))
print("Output adapter/checkpoints:", OUTPUT_DIR)
print(f"Config: epochs={EPOCHS}, batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")


In [ ]:
# 9. Start fine-tuning
# This can take hours. Keep the computer powered and monitor GPU memory.
# The noisy val set is the real checkpoint-selection signal. With the A5000, batch 4 should fit. If you see CUDA out of memory, set
# BATCH_SIZE=1 and GRAD_ACCUM=4 in the previous cell, then rerun this cell.
trainer = train_sarvam_lora(
    train_jsonl_path=str(TRAIN_PATH),
    val_jsonl_path=str(VAL_PATH),
    output_dir=str(OUTPUT_DIR),
    base_model_id=BASE_MODEL_ID,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    gradient_accumulation_steps=GRAD_ACCUM,
)

print("Training function returned:", type(trainer))
print("Training output directory:", OUTPUT_DIR)


In [ ]:
# 10. List saved outputs
if OUTPUT_DIR.exists():
    files = [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]
    if not files:
        print("No files found yet. Confirm that Sarvam.py saves the adapter/checkpoints.")
    for p in files:
        print(f"{p.relative_to(WORK_DIR)} | {p.stat().st_size / (1024**2):.2f} MB")


In [ ]:
# 10B. Verify the adapter is self-contained BEFORE copying it back.
# Run this in a FRESH kernel after training: the training model may still hold VRAM.
# It only needs cell 2 (torch); this cell sets up paths and imports the workspace itself.
import json
import sys
from pathlib import Path
from transformers import AutoTokenizer

WORK_DIR = Path.cwd() / "Punarvaani_training"
OUTPUT_DIR = WORK_DIR / "checkpoints" / "sarvam_lora"
sys.path.insert(0, str(WORK_DIR))
import Models.Sarvam as sarvam_module

EXPECTED_TAGS = ["<S2S>", "<PHONEMES>", "</PHONEMES>", "<TEXT>"]

adapter_file = OUTPUT_DIR / "adapter_model.safetensors"
if not adapter_file.exists():
    adapter_file = OUTPUT_DIR / "adapter_model.bin"
if not adapter_file.exists():
    print("No adapter file at the root. Check OUTPUT_DIR/checkpoint-*/ and copy the latest one's contents into OUTPUT_DIR.")
    raise FileNotFoundError("No adapter_model.safetensors/.bin found. Did training finish and save?")

size_mb = adapter_file.stat().st_size / 1024**2
cfg = json.loads((OUTPUT_DIR / "adapter_config.json").read_text(encoding="utf-8"))
print("adapter file:", adapter_file.name, f"{size_mb:.1f} MB")
print("base model:", cfg.get("base_model_name_or_path"))
print("modules_to_save:", cfg.get("modules_to_save"))

if size_mb < 100:
    raise RuntimeError(
        "This looks like a LoRA-only adapter (under 100 MB). The four tag embeddings are missing. "
        "Re-run training with modules_to_save=['embed_tokens','lm_head'] and do not copy this folder."
    )

tok = AutoTokenizer.from_pretrained(str(OUTPUT_DIR), use_fast=True)
print("tag token lengths:", {t: len(tok.encode(t, add_special_tokens=False)) for t in EXPECTED_TAGS})

from Models.Sarvam import HindiPhonemeTranslator
translator = HindiPhonemeTranslator(adapter_path=str(OUTPUT_DIR))
print("MOCK:", translator.is_mock, translator.mock_reason)
print("sample translation:", translator.translate("N A M A S TD E | BH AA R A TD"))


In [ ]:
# 10C. Zip only the adapter files (not the checkpoint-XXXX folders).
import zipfile

zip_path = WORK_DIR / "sarvam_lora_for_repo.zip"
names = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "adapter_model.bin",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "added_tokens.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in names:
        p = OUTPUT_DIR / name
        if p.exists():
            z.write(p, arcname=name)
            print("added:", name)

print("Created:", zip_path)
print("Copy this zip to the repo PC and unzip it into checkpoints/sarvam_lora/.")


## 11. Optional: transfer files from Colab if you don't have them on Windows

In your **Colab** notebook, download the files one by one:

```python
from google.colab import files
files.download('/content/Punarvaani/Models/Sarvam.py')
files.download('/content/Punarvaani/sarvam_data/train.jsonl')
files.download('/content/Punarvaani/sarvam_data/val.jsonl')
```

Then select those downloaded files in Section 3 above.

If the dataset downloads are too large for your browser, zip them in Colab first:
```python
import shutil
shutil.make_archive('/content/punarvaani_sarvam_data', 'zip',
                    root_dir='/content/Punarvaani/sarvam_data')
from google.colab import files
files.download('/content/punarvaani_sarvam_data.zip')
```
Unzip locally and select `train.jsonl` and `val.jsonl` from the extracted folder.


## 12. Push code to GitHub (optional; this notebook itself does not clone)

This workspace is independent of Git. If you want to push your changes later:
- Open GitHub and upload the changed `Sarvam.py` through the web UI, or
- Download the ZIP of the repo, copy the modified file into that folder, then use GitHub Desktop to commit/push.

**Do not upload** dataset files, model checkpoints, adapter weights, tokens, or `.venv` to a public repository unless you have checked their privacy, size, and license implications.

Suggested `.gitignore` entries:
```
.venv/
__pycache__/
*.pyc
checkpoints/
*.safetensors
*.bin
sarvam_data/*.jsonl
```

## Troubleshooting
- **GPU unavailable:** install correct NVIDIA driver and CUDA-enabled PyTorch, restart Jupyter.
- **CUDA out of memory:** batch size 1; smaller sequence length if allowed by your code; use gradient checkpointing/quantization only if your implementation supports them.
- **`train_sarvam_lora` missing:** selected the wrong `Sarvam.py`; select the correct file.
- **Hugging Face access denied:** authenticate and confirm access to `sarvamai/sarvam-1`.
- **Notebook kernel wrong:** launch Jupyter from the activated venv and choose the matching kernel.
- **Checkpoint not saved:** inspect your `Sarvam.py` save logic and ensure output directory is used correctly.
